# Did the transcoder learn the right computation?

This notebook tests the suspected failure mechanisms using the **existing checkpoint**. It does not retrain, select steering features, or execute modified actions. All experiment definitions are literal notebook cells; diagnostics run directly in this kernel with native tracebacks. No repository checkout or Python experiment subprocess is used. Package installation uses external setup tools with live output.

| Test | Matched comparison / evidence |
|---|---|
| Did updates occur? | Saved optimizer step counters, training metrics, finite dictionary weights, decoder norms |
| Wrong collection distribution? | Native inference vs fresh noise at identical scheduled times vs fresh noise at training-distribution sampled times |
| Is it actually sparse? | Active features/token and observed never-fired fractions; no unsupported dead-feature claim |
| Does replacement preserve actions? | Original vs individual layers 0/8/17 vs all checkpoint layers; identical starting noise |
| Is the measurement trustworthy? | Original/original repeat, first-step noise-grid identity, exact record counts, streaming statistics |

Default cohort: **tasks 0, 7, 8, 9 × initial states 0, 1, 2 × 1 noise draw = 12 cases**. Each case uses its original task instruction and initial images. All checkpoint layers are measured over 10 denoising calls. The two fresh-noise conditions are diagnostic computations, not robot policies. Only normal-inference and replacement action predictions are compared. No action is executed in the simulator.

Use `PHASE="metadata"` for a CPU checkpoint inspection first. Use `PHASE="all"` on a high-RAM L4/A100 for the complete diagnostic. A fresh runtime may require **one session restart after installation**; rerun from Controls afterward. Keep the same run name to resume complete cases. Change the run name if the checkpoint, cohort, definitions, policy snapshot, or runtime changes.

The collection timer begins after loading. It stops between complete cases; it is not a hard wall-clock guarantee. Saved results are conditional on completed cases when partial. Report reconstruction by layer/time and action error against the repeat floor. No universal pass threshold is assumed. These scenes are **not established as held out from pretraining**, and this is not attribution validation, task success, steerability, or collision prevention.


In [ ]:
# @title Controls
from pathlib import Path
DRIVE_ROOT = '/content/drive/MyDrive/groot-run-shared-programmer908' # @param {type:"string"}
CHECKPOINT = '' # @param {type:"string"}
RUN_NAME = 'transcoder-training-audit-001' # @param {type:"string"}
PHASE = 'all' # @param ["metadata", "all"]
TASK_IDS = '0,7,8,9' # @param {type:"string"}
INIT_STATE_IDS = '0,1,2' # @param {type:"string"}
NOISE_DRAWS = 1 # @param {type:"integer"}
INFERENCE_STEPS = 10 # @param {type:"integer"}
REPLACEMENT_LAYERS = '0,8,17' # @param {type:"string"}
REPLACE_ALL = True # @param {type:"boolean"}
COLLECTION_MINUTES = 30 # @param {type:"integer"}
HF_OFFLINE = True # @param {type:"boolean"}
ALLOW_AUTH_REFRESH = True # @param {type:"boolean"}
POLICY_REPO = 'lerobot/pi05_libero_finetuned'
POLICY_REVISION = 'main' # resolved snapshot commit is persisted in the manifest
if PHASE not in ('metadata','all') or not RUN_NAME or Path(RUN_NAME).name!=RUN_NAME:
    raise ValueError('Use phase metadata/all and a single nonempty run-folder name')


In [ ]:
# @title Mount Existing Drive Folder
from google.colab import drive
from pathlib import Path
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
DRIVE_ROOT = Path(DRIVE_ROOT)
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f'{DRIVE_ROOT} is not visible. Use the same Drive shortcut/path as the working steering notebook.')
RUN_DIR = DRIVE_ROOT/'outputs/transcoder-audit'/RUN_NAME
CHECKPOINT_PATH = Path(CHECKPOINT) if CHECKPOINT.strip() else DRIVE_ROOT/'transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt'
if not CHECKPOINT_PATH.is_absolute(): CHECKPOINT_PATH=DRIVE_ROOT/CHECKPOINT_PATH
if not CHECKPOINT_PATH.is_file(): raise FileNotFoundError(CHECKPOINT_PATH)
print('Checkpoint:', CHECKPOINT_PATH)
print('Size (GiB):', round(CHECKPOINT_PATH.stat().st_size/2**30,2))
print('Saved evidence:', RUN_DIR)


In [ ]:
# @title Reuse Or Install The Pinned Runtime
if PHASE == "all":
    if not Path("/dev/nvidia0").exists():
        raise RuntimeError("Select a high-RAM L4/A100 GPU runtime, or use PHASE=metadata.")
    # @title Install Pinned LeRobot Runtime (Repairs Existing Environment)

    import os
    import sys
    import subprocess
    from pathlib import Path

    # Rerunning this cell cannot unload native modules. Avoid a reinstall loop.
    install_receipt = Path('/content/pi05-audit-install-receipt.json')
    if install_receipt.exists():
        import json
        if json.loads(install_receipt.read_text()).get('pid') == os.getpid():
            raise RuntimeError('Packages are already installed. Use Runtime > Restart session once, then Run all; do not delete the runtime.')

    # Set cache and rendering variables before importing Transformers or LeRobot;
    # these libraries cache environment settings at import time in the kernel.
    os.environ.update({
        "HF_HOME": "/content/hf_home",
        "HF_HUB_CACHE": "/content/hf_home/hub",
        "MUJOCO_GL": "egl",
        "PYOPENGL_PLATFORM": "egl",
        "MUJOCO_EGL_DEVICE_ID": "0",
        "MPLBACKEND": "Agg",
        "LIBERO_CONFIG_PATH": "/content/pi05-audit-support/.libero",
        "LIBERO_DATASET_DIR": "/content/pi05-audit-support/data/libero/datasets",
    })
    for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
        if HF_OFFLINE:
            os.environ[key] = "1"
        else:
            os.environ.pop(key, None)
    loaded_hub = sys.modules.get("huggingface_hub.constants")
    if loaded_hub is not None and loaded_hub.HF_HUB_CACHE != os.environ["HF_HUB_CACHE"]:
        raise RuntimeError("Restart the Colab session, then rerun from the top to use the configured model cache.")

    PYTHON = Path(sys.executable)  # Install into the kernel used for direct collection.
    UV_BIN_DIR = Path("/content/uv-bin")
    UV = str(UV_BIN_DIR / "uv")


    def run(cmd, *, env=None):
        # External setup tools (uv/apt/git), never a Python collection worker.
        # Stream stderr too: inherited child output may be absent from Colab's cell.
        from collections import deque
        import shlex
        cmd = list(map(str, cmd))
        print("$", shlex.join(cmd), flush=True)
        tail = deque(maxlen=100)
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if returncode:
            detail = "".join(tail).strip() or "The command produced no output."
            raise RuntimeError(f"Setup command failed (exit {returncode}): {shlex.join(cmd)}\n\n{detail}")
        return subprocess.CompletedProcess(cmd, returncode)


    def install_uv() -> str:
        # Colab may already have /usr/local/bin/uv, but its version/behavior can differ.
        # Use a notebook-owned binary so every clean runtime follows the same path.
        UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
        installer = Path("/tmp/install-uv.sh")
        run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
        env = os.environ.copy()
        env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
        run(["sh", installer], env=env)
        os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
        run([UV, "--version"])
        return UV


    # Check in this kernel; do not mix a Python 3.12 venv into a Python 3.13 kernel.
    # Package installation cannot replace native modules already loaded by this process.
    loaded_runtime = any(name in sys.modules for name in ("torch", "numpy", "transformers", "lerobot"))
    try:
        import importlib.util
        from importlib.metadata import version
        assert version('lerobot') == '0.4.4'
        assert version('torch').split('+')[0] == '2.10.0'
        assert version('torchvision').split('+')[0] == '0.25.0'
        from transformers.models.siglip import check
        assert check.check_whether_transformers_replace_is_installed_correctly()
        import torch
        assert torch.cuda.is_available()
        assert all(importlib.util.find_spec(name) for name in ('libero', 'langfuse', 'sklearn', 'joblib', 'PIL', 'matplotlib'))
        ready = True
    except (ImportError, AssertionError, ValueError, RuntimeError) as exc:
        print("Runtime needs installation:", exc)
        ready = False
    if ready:
        print("Reusing existing CUDA/LeRobot runtime:", PYTHON)
        if not Path(UV).exists():
            UV = install_uv()
    else:
        apt_packages = [
            "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
            "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
            "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
        ]
        apt_env = os.environ.copy()
        apt_env["DEBIAN_FRONTEND"] = "noninteractive"
        run(["apt-get", "update", "-qq"], env=apt_env)
        run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
        UV = install_uv()

        # egl-probe's source build predates CMake 4. Install its build tools first.
        run([UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
             "cmake==3.31.6", "setuptools>=71,<81", "wheel"])
        import sysconfig
        os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
        run(["cmake", "--version"])
        run([UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
             "--no-build-isolation-package", "egl-probe", "egl-probe==1.0.2"])

        # Match scripts/pi05_transformers_overrides.txt; the repo is cloned in the next cell.
        transformers_override = Path("/content/pi05-transformers-overrides.txt")
        transformers_override.write_text('transformers @ git+https://github.com/huggingface/transformers.git@dcddb970176382c0fcf4521b0c0e6fc15894dfe0' + "\n")

        run([
            UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
            "--torch-backend", "cu128",
            "--overrides", str(transformers_override),
            "--no-build-isolation-package", "egl-probe", "cmake==3.31.6",
            "lerobot[libero]==0.4.4", "hf-transfer", "opencv-python", "numpy", "langfuse",
            "scikit-learn", "joblib", "pillow", "matplotlib", "torch==2.10.0", "torchvision==0.25.0",
        ])

        import json
        install_receipt.write_text(json.dumps({'pid': os.getpid()}))
        print("Installed into notebook Python:", PYTHON)
        # The readiness probe itself may also have imported modules before failing.
        if loaded_runtime or any(name in sys.modules for name in ("torch", "numpy", "transformers", "lerobot")):
            raise RuntimeError("Packages installed. Restart the Colab session once, then rerun the notebook. "
                               "This clears old Python/native modules; Drive caches are preserved.")

else:
    import torch, numpy
    print("Using current torch for metadata inspection only; no package replacement.")


In [ ]:
# @title Prepare Model Cache And LIBERO Assets (Skipped For Metadata Only)
import os, shutil, site, tarfile
from pathlib import Path
snapshots = {}
if PHASE == 'all':
    cache=Path('/content/hf_home'); cache.mkdir(parents=True,exist_ok=True)
    archive=DRIVE_ROOT/'archives/hf_home.tar'
    policy_cache=cache/'hub'/('models--'+POLICY_REPO.replace('/','--'))/'snapshots'
    policy_cached=policy_cache.is_dir() and any((p/'config.json').is_file() and
        ((p/'model.safetensors').is_file() or (p/'model.safetensors.index.json').is_file()) for p in policy_cache.iterdir())
    if not policy_cached:
        if archive.is_file():
            print('Restoring model cache archive',flush=True)
            with tarfile.open(archive) as handle: handle.extractall(cache,filter='data')
        elif (DRIVE_ROOT/'hf_home').is_dir():
            print('Copying existing Drive model cache',flush=True)
            shutil.copytree(DRIVE_ROOT/'hf_home',cache,dirs_exist_ok=True)
    from huggingface_hub import snapshot_download, constants as hf_constants
    for repo,kind in [(POLICY_REPO,'model'),('google/paligemma-3b-pt-224','model'),('lerobot/libero-assets','dataset')]:
        revision=POLICY_REVISION if repo==POLICY_REPO else 'main'
        try:
            path=snapshot_download(repo,repo_type=kind,revision=revision,local_files_only=True)
        except Exception:
            if not ALLOW_AUTH_REFRESH: raise
            token=os.environ.get('HF_TOKEN')
            if not token:
                from google.colab import userdata
                try: token=userdata.get('HF_TOKEN')
                except Exception: token=None
            old=hf_constants.HF_HUB_OFFLINE; hf_constants.HF_HUB_OFFLINE=False
            try: path=snapshot_download(repo,repo_type=kind,revision=revision,token=token,local_files_only=False)
            finally: hf_constants.HF_HUB_OFFLINE=old
        snapshots[repo]=str(Path(path).resolve())
        print(repo,snapshots[repo],flush=True)
    roots=[Path(p)/'libero/libero' for p in site.getsitepackages()]
    libero_root=next((p for p in roots if p.is_dir()),None)
    if libero_root is None: raise RuntimeError('hf-libero is missing; rerun runtime setup')
    assets=libero_root/'assets'
    if not (assets/'scenes/libero_tabletop_base_style.xml').exists():
        shutil.copytree(snapshots['lerobot/libero-assets'],assets,dirs_exist_ok=True)
    if not (assets/'scenes/libero_tabletop_base_style.xml').exists(): raise RuntimeError('LIBERO assets are incomplete')
    Path('/content/pi05-audit-support').mkdir(exist_ok=True)
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime for PHASE=all')
    ram_gib=int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')))/2**20
    gpu_gib=torch.cuda.get_device_properties(0).total_memory/2**30
    print(f'GPU memory: {gpu_gib:.1f} GiB; system memory: {ram_gib:.1f} GiB')
    if gpu_gib<21 or ram_gib<23: raise RuntimeError('Use a high-RAM L4/A100: at least 22 GiB GPU and 24 GiB system RAM recommended')
else:
    print('Metadata-only: no simulator, policy, assets or GPU required.')


In [ ]:
# @title Definitions 1 — Exact Dictionary And Policy Guards
"""DifFRACT-style timestep-conditioned transcoders.

The transcoder approximates one transformer MLP:

    TC_l(x, t) ~= MLP_l(x)

where x is one token-position hidden vector and t is the flow/diffusion
timestep. The module also supports batched token tensors shaped
``[batch, tokens, d_model]``.
"""


from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import Tensor, nn


@dataclass(frozen=True)
class TimeConditionedTranscoderConfig:
    """Configuration for one timestep-conditioned transcoder."""

    d_model: int
    d_features: int | None = None
    expansion_factor: int = 16
    time_embedding_dim: int | None = None
    time_hidden_dim: int | None = None
    min_period: float = 4e-3
    max_period: float = 4.0
    eps: float = 1e-8

    @property
    def latent_dim(self) -> int:
        return self.d_features if self.d_features is not None else self.d_model * self.expansion_factor

    @property
    def time_dim(self) -> int:
        return self.time_embedding_dim if self.time_embedding_dim is not None else self.d_model

    @property
    def time_hidden(self) -> int:
        return self.time_hidden_dim if self.time_hidden_dim is not None else self.d_model


@dataclass(frozen=True)
class TranscoderLoss:
    """Scalar loss terms for logging and optimization."""

    total: Tensor
    normalized_mse: Tensor
    l1: Tensor


def sinusoidal_time_embedding(
    timesteps: Tensor,
    dim: int,
    *,
    min_period: float = 4e-3,
    max_period: float = 4.0,
) -> Tensor:
    """Create sinusoidal timestep embeddings for scalar flow/diffusion times.

    Args:
        timesteps: Tensor shaped ``[batch]`` or ``[batch, 1]``.
        dim: Output embedding width.
        min_period: Shortest sinusoid period.
        max_period: Longest sinusoid period.
    """
    if dim <= 0:
        raise ValueError(f"dim must be positive, got {dim}")

    t = timesteps.float().reshape(-1, 1)
    half_dim = dim // 2
    if half_dim == 0:
        return t.new_zeros((t.shape[0], dim))

    periods = torch.logspace(
        start=torch.log10(t.new_tensor(min_period)),
        end=torch.log10(t.new_tensor(max_period)),
        steps=half_dim,
        device=t.device,
        dtype=t.dtype,
    )
    phases = 2.0 * torch.pi * t / periods.unsqueeze(0)
    emb = torch.cat([torch.sin(phases), torch.cos(phases)], dim=-1)
    if emb.shape[-1] < dim:
        emb = F.pad(emb, (0, dim - emb.shape[-1]))
    return emb


def normalized_mse_loss(prediction: Tensor, target: Tensor, variance_denominator: Tensor, eps: float) -> Tensor:
    """Return variance-normalized squared reconstruction error.

    ``variance_denominator`` is the per-transcoder denominator:
    ``sum_j Var[MLP_l(x)_j]`` estimated over that MLP's activation records.
    """
    squared_error = (prediction.float() - target.float()).pow(2).sum(dim=-1).mean()
    denom = variance_denominator.float().clamp_min(eps)
    return squared_error / denom


class TimeConditionedTranscoder(nn.Module):
    """Sparse timestep-conditioned transcoder for one MLP."""

    def __init__(self, config: TimeConditionedTranscoderConfig):
        super().__init__()
        self.config = config
        d_model = config.d_model
        d_features = config.latent_dim

        self.time_mlp = nn.Sequential(
            nn.Linear(config.time_dim, config.time_hidden),
            nn.SiLU(),
            nn.Linear(config.time_hidden, 2 * d_model),
        )
        self.encoder = nn.Linear(d_model, d_features)
        self.decoder = nn.Linear(d_features, d_model)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.encoder.weight, a=5**0.5)
        nn.init.zeros_(self.encoder.bias)
        nn.init.kaiming_uniform_(self.decoder.weight, a=5**0.5)
        nn.init.zeros_(self.decoder.bias)

        last_time = self.time_mlp[-1]
        if not isinstance(last_time, nn.Linear):
            raise TypeError("Expected final time_mlp module to be nn.Linear")
        nn.init.zeros_(last_time.weight)
        nn.init.zeros_(last_time.bias)

    def _time_scale_shift(self, timesteps: Tensor, *, dtype: torch.dtype, device: torch.device) -> tuple[Tensor, Tensor]:
        time_emb = sinusoidal_time_embedding(
            timesteps.to(device=device),
            self.config.time_dim,
            min_period=self.config.min_period,
            max_period=self.config.max_period,
        ).to(dtype=dtype)
        scale_shift = self.time_mlp(time_emb)
        scale, shift = scale_shift.chunk(2, dim=-1)
        return scale, shift

    def encode(self, x: Tensor, timesteps: Tensor) -> tuple[Tensor, Tensor]:
        """Return ``(feature_preactivation, sparse_latent)``.

        Args:
            x: MLP input, shaped ``[records, d_model]`` or
                ``[batch, tokens, d_model]``.
            timesteps: Scalar timesteps shaped ``[records]`` for 2D x or
                ``[batch]`` for 3D x.
        """
        if x.shape[-1] != self.config.d_model:
            raise ValueError(f"Expected x last dim {self.config.d_model}, got {x.shape[-1]}")

        param_dtype = self.encoder.weight.dtype
        x_for_transcoder = x.to(dtype=param_dtype)
        scale, shift = self._time_scale_shift(timesteps, dtype=param_dtype, device=x.device)
        if x.ndim == 3:
            if scale.shape[0] != x.shape[0]:
                raise ValueError(f"Expected {x.shape[0]} timesteps for 3D x, got {scale.shape[0]}")
            scale = scale[:, None, :]
            shift = shift[:, None, :]
        elif x.ndim == 2:
            if scale.shape[0] != x.shape[0]:
                raise ValueError(f"Expected {x.shape[0]} timesteps for 2D x, got {scale.shape[0]}")
        else:
            raise ValueError(f"Expected x with rank 2 or 3, got shape {tuple(x.shape)}")

        x_mod = x_for_transcoder * (1.0 + scale) + shift
        preactivation = self.encoder(x_mod)
        z = F.relu(preactivation)
        return preactivation, z

    def forward(
        self,
        x: Tensor,
        timesteps: Tensor,
        *,
        return_preactivation: bool = False,
    ) -> tuple[Tensor, Tensor] | tuple[Tensor, Tensor, Tensor]:
        """Return ``(predicted_mlp_output, sparse_latent)``.

        Set ``return_preactivation=True`` when circuit tracing needs the
        encoder pre-ReLU feature values as attribution targets.
        """
        preactivation, z = self.encode(x, timesteps)
        y_hat = self.decoder(z)
        if return_preactivation:
            return y_hat, z, preactivation
        return y_hat, z

    def loss(
        self,
        x: Tensor,
        timesteps: Tensor,
        target: Tensor,
        variance_denominator: Tensor,
        *,
        lambda_l1: float,
    ) -> TranscoderLoss:
        """Compute DifFRACT-style normalized reconstruction + sparsity loss."""
        prediction, latent = self(x, timesteps)
        normalized_mse = normalized_mse_loss(
            prediction,
            target,
            variance_denominator,
            eps=self.config.eps,
        )
        l1 = latent.float().abs().sum(dim=-1).mean()
        total = normalized_mse + lambda_l1 * l1
        return TranscoderLoss(total=total, normalized_mse=normalized_mse, l1=l1)

    @torch.no_grad()
    def renormalize_decoder_columns(self, eps: float | None = None) -> None:
        """Normalize decoder feature vectors after an optimizer step.

        ``nn.Linear`` stores decoder weights as ``[d_model, d_features]``.
        Each column is the output vector for one sparse feature.
        """
        epsilon = self.config.eps if eps is None else eps
        weight = self.decoder.weight
        column_norms = weight.norm(dim=0, keepdim=True).clamp_min(epsilon)
        weight.div_(column_norms)

"""Refuse to run on a Pi0.5 whose pretrained weights did not actually load.

LeRobot's ``PI05Policy.from_pretrained`` wraps the whole state-dict load in a
``try``/``except`` that prints ``Warning: Could not load state dict`` and
returns the freshly initialised model anyway; and with ``strict=False`` a
partially matching checkpoint loads silently with missing keys. Either way the
policy runs on random weights and every downstream number is meaningless, while
nothing in the process exits non-zero.

The fix is to record what ``load_state_dict`` reported and check it after the
policy is built. A policy with no record was never loaded; a policy with
missing keys is not the checkpoint.
"""


import copy
from typing import Any

LOAD_REPORT_ATTR = "_pi05_mi_load_report"


def restore_tied_parameter_aliases(model, state_dict):
    """Restore omitted safetensors keys only for parameters the model actually shares.

    PaliGemma ties its token embedding to lm_head. Safetensors may save only
    one name; nn.Module.load_state_dict(strict=True) still requires both.
    Never invent a tensor or infer a tie from matching names or shapes.
    """
    groups = {}
    for name, parameter in model.named_parameters(remove_duplicate=False):
        groups.setdefault(id(parameter), []).append(name)
    restored = {}
    result = state_dict
    for names in groups.values():
        present = [name for name in names if name in state_dict]
        if not present:
            continue
        for name in names:
            if name not in state_dict:
                if result is state_dict:
                    result = copy.copy(state_dict)
                result[name] = state_dict[present[0]]
                restored[name] = present[0]
    return result, restored


def install_load_recorder(cls: type | None = None) -> None:
    """Make ``cls.load_state_dict`` leave a record of missing and unexpected keys.

    Defaults to LeRobot's ``PI05Policy``. Idempotent.
    """
    if cls is None:
        from lerobot.policies.pi05 import modeling_pi05

        cls = modeling_pi05.PI05Policy
    original = cls.load_state_dict
    if getattr(original, "_pi05_mi_recorder", False):
        return

    def load_state_dict_recording(self, state_dict, strict: bool = True, assign: bool = False):
        state_dict, restored = restore_tied_parameter_aliases(self, state_dict)
        # Clear any previous successful report before a later load can fail.
        setattr(self, LOAD_REPORT_ATTR, None)
        try:
            result = original(self, state_dict, strict=strict, assign=assign)
        except Exception as exc:
            setattr(self, LOAD_REPORT_ATTR, {'error': f'{type(exc).__name__}: {exc}'})
            raise
        setattr(
            self,
            LOAD_REPORT_ATTR,
            {
                "missing": list(getattr(result, "missing_keys", [])),
                "unexpected": list(getattr(result, "unexpected_keys", [])),
                "provided": len(state_dict),
                "strict": bool(strict),
                "restored_tied_aliases": restored,
            },
        )
        return result

    load_state_dict_recording._pi05_mi_recorder = True  # type: ignore[attr-defined]
    cls.load_state_dict = load_state_dict_recording


def load_report(policy: Any) -> dict[str, Any] | None:
    return getattr(policy, LOAD_REPORT_ATTR, None)


def assert_weights_loaded(policy: Any, *, source: str = "", max_listed: int = 5) -> dict[str, Any]:
    """Raise unless the policy's pretrained weights loaded completely.

    Unexpected keys (present in the checkpoint, absent from the model) are
    tolerated and returned in the report; missing keys are not, because a
    missing parameter is one running on its random initialisation.
    """
    report = load_report(policy)
    where = f" from {source}" if source else ""
    if report is None:
        raise RuntimeError(
            f"Pi0.5 weights were never loaded{where}. LeRobot swallowed the failure and returned a "
            "randomly initialised model; look for 'Could not load state dict' in the log above. "
            "Refusing to run: every measurement would be about random weights."
        )
    if report.get('error'):
        raise RuntimeError(
            f"Pi0.5 checkpoint loading failed{where}:\n{report['error']}\n"
            "Refusing to run with incomplete weights."
        )
    missing = report["missing"]
    if missing:
        shown = "\n  ".join(missing[:max_listed])
        more = f"\n  ... and {len(missing) - max_listed} more" if len(missing) > max_listed else ""
        raise RuntimeError(
            f"{len(missing)} Pi0.5 parameters did not load{where} and are running on random "
            f"initialisation:\n  {shown}{more}\n"
            "A vision-tower key layout mismatch is the usual cause. Refusing to run."
        )
    return report

"""Runtime wrappers for Pi0.5 action-expert MLP transcoders."""


import re
import types
from collections import defaultdict
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Callable, Iterator, Literal

import torch
from torch import Tensor, nn



TranscoderMode = Literal["train", "probe", "replace"]

ACTION_EXPERT_MLP_RE = re.compile(r"paligemma_with_expert\.gemma_expert\.model\.layers\.(\d+)\.mlp$")


@dataclass(frozen=True)
class MLPActivationRecord:
    """One captured MLP input-output tensor for one layer call."""

    name: str
    layer_index: int
    x: Tensor
    y: Tensor
    timestep: Tensor


@dataclass(frozen=True)
class MLPTranscoderLatentRecord:
    """Compact latent summary for one transcoder call."""

    name: str
    layer_index: int
    mode: str
    timestep: Tensor
    latent_shape: tuple[int, ...]
    latent_l0: Tensor
    latent_l1: Tensor
    latent_max: Tensor
    top_indices: Tensor
    top_values: Tensor
    full_latent: Tensor | None = None


@dataclass(frozen=True)
class MLPTranscoderTraceRecord:
    """Gradient-preserving transcoder tensors from one wrapped MLP call."""

    name: str
    layer_index: int
    mode: str
    timestep: Tensor
    preactivation: Tensor
    latent: Tensor


@dataclass(frozen=True)
class ActionExpertMLPTarget:
    """One Pi0.5 action-expert MLP target."""

    name: str
    layer_index: int
    module: nn.Module
    d_model: int


class Pi05TranscoderContext:
    """Shared runtime state for Pi0.5 transcoder wrappers."""

    def __init__(
        self,
        mode: TranscoderMode = "train",
        *,
        detach_records: bool = True,
        capture_records: bool = True,
        capture_latents: bool = False,
        capture_traces: bool = False,
        latent_top_k: int = 64,
        save_full_latents: bool = False,
        latent_callback: Callable[[str, int, Tensor, Tensor], None] | None = None,
        latent_intervention: Callable[[str, int, Tensor, Tensor], Tensor | None] | None = None,
        trace_callback: Callable[[str, int, Tensor, Tensor, Tensor], None] | None = None,
        diffusion_callback: Callable[[str, Tensor, Tensor | None], None] | None = None,
        store_latent_summaries: bool = True,
        capture_diffusion: bool = False,
    ):
        self.mode = mode
        self.detach_records = detach_records
        self.capture_records = capture_records
        self.capture_latents = capture_latents
        self.capture_diffusion = capture_diffusion
        self.capture_traces = capture_traces
        self.latent_top_k = latent_top_k
        self.save_full_latents = save_full_latents
        self.latent_callback = latent_callback
        self.trace_callback = trace_callback
        # Causal intervention hook. In replace mode the wrapper offers each
        # transcoder latent to this callable; returning a tensor substitutes it
        # and the decoder is re-run, so the patched features -- and nothing
        # else -- drive the rest of the forward pass.
        self.latent_intervention = latent_intervention
        self.diffusion_callback = diffusion_callback
        self.store_latent_summaries = store_latent_summaries
        self.current_timestep: Tensor | None = None
        self.records: dict[str, list[MLPActivationRecord]] = defaultdict(list)
        self.latents: dict[str, list[MLPTranscoderLatentRecord]] = defaultdict(list)
        self.trace_records: dict[str, list[MLPTranscoderTraceRecord]] = defaultdict(list)

    @contextmanager
    def use_timestep(self, timestep: Tensor) -> Iterator[None]:
        previous = self.current_timestep
        self.current_timestep = timestep
        try:
            yield
        finally:
            self.current_timestep = previous

    def clear_records(self) -> None:
        self.records.clear()
        self.latents.clear()
        self.trace_records.clear()

    def timestep_for(self, x: Tensor) -> Tensor:
        if self.current_timestep is None:
            raise RuntimeError(
                "No Pi0.5 timestep is active. Call the wrapped model through PI05Pytorch.forward "
                "or PI05Pytorch.denoise_step so the transcoder context can see t."
            )
        timestep = self.current_timestep.to(device=x.device)
        if timestep.ndim == 0:
            timestep = timestep.reshape(1)
        if x.ndim == 3 and timestep.shape[0] != x.shape[0]:
            if timestep.numel() == 1:
                timestep = timestep.expand(x.shape[0])
            else:
                raise ValueError(f"Expected timestep batch {x.shape[0]}, got shape {tuple(timestep.shape)}")
        if x.ndim == 2 and timestep.shape[0] != x.shape[0]:
            if timestep.numel() == 1:
                timestep = timestep.expand(x.shape[0])
            else:
                raise ValueError(f"Expected one timestep per record for x shape {tuple(x.shape)}, got {tuple(timestep.shape)}")
        return timestep

    def record(self, name: str, layer_index: int, x: Tensor, y: Tensor, timestep: Tensor) -> None:
        if not self.capture_records:
            return
        if self.detach_records:
            x = x.detach()
            y = y.detach()
            timestep = timestep.detach()
        self.records[name].append(
            MLPActivationRecord(
                name=name,
                layer_index=layer_index,
                x=x,
                y=y,
                timestep=timestep,
            )
        )

    def record_trace(
        self,
        name: str,
        layer_index: int,
        preactivation: Tensor,
        latent: Tensor,
        timestep: Tensor,
    ) -> None:
        if not self.capture_traces:
            return
        if self.trace_callback is not None:
            self.trace_callback(name, layer_index, preactivation, latent, timestep)
            return
        self.trace_records[name].append(
            MLPTranscoderTraceRecord(
                name=name,
                layer_index=layer_index,
                mode=self.mode,
                timestep=timestep,
                preactivation=preactivation,
                latent=latent,
            )
        )

    def record_latent(self, name: str, layer_index: int, latent: Tensor, timestep: Tensor) -> None:
        if not self.capture_latents:
            return
        if self.latent_callback is not None:
            self.latent_callback(name, layer_index, latent, timestep)
        if not self.store_latent_summaries:
            return
        with torch.no_grad():
            z = latent.detach().float()
            timestep = timestep.detach().float().cpu()
            latent_l0 = (z > 0).sum(dim=-1).detach().cpu()
            latent_l1 = z.abs().sum(dim=-1).detach().cpu()
            latent_max = z.max(dim=-1).values.detach().cpu()

            flat = z.reshape(-1)
            top_k = min(max(0, int(self.latent_top_k)), flat.numel())
            if top_k > 0:
                top_values, top_flat_indices = torch.topk(flat, k=top_k)
                top_indices = torch.stack(torch.unravel_index(top_flat_indices.cpu(), z.shape), dim=-1)
            else:
                top_values = torch.empty(0, dtype=torch.float32)
                top_indices = torch.empty((0, z.ndim), dtype=torch.long)

            full_latent = z.cpu() if self.save_full_latents else None
            self.latents[name].append(
                MLPTranscoderLatentRecord(
                    name=name,
                    layer_index=layer_index,
                    mode=self.mode,
                    timestep=timestep,
                    latent_shape=tuple(z.shape),
                    latent_l0=latent_l0,
                    latent_l1=latent_l1,
                    latent_max=latent_max,
                    top_indices=top_indices.cpu(),
                    top_values=top_values.cpu(),
                    full_latent=full_latent,
                )
            )

    def record_diffusion(self, kind: str, value: Tensor, timestep: Tensor | None = None) -> None:
        if not self.capture_diffusion or self.diffusion_callback is None:
            return
        self.diffusion_callback(kind, value, timestep)


class WrappedActionExpertMLP(nn.Module):
    """Wrapper that preserves, observes, or replaces one Pi0.5 action-expert MLP."""

    def __init__(
        self,
        *,
        name: str,
        layer_index: int,
        original_mlp: nn.Module,
        context: Pi05TranscoderContext,
        transcoder: TimeConditionedTranscoder | None = None,
    ):
        super().__init__()
        self.name = name
        self.layer_index = layer_index
        self.original_mlp = original_mlp
        self.transcoder = transcoder
        self.context = context

    def forward(self, x: Tensor) -> Tensor:
        timestep = self.context.timestep_for(x)

        if self.context.mode == "replace":
            if self.transcoder is None:
                raise RuntimeError(f"Cannot run {self.name} in replace mode without a transcoder")
            y_hat, latent, preactivation = self.transcoder(x, timestep, return_preactivation=True)
            intervention = self.context.latent_intervention
            if intervention is not None:
                if self.context.capture_traces:
                    raise RuntimeError(
                        f"{self.name}: cannot capture traces and intervene in the same pass. "
                        "Attribution needs the unmodified computational graph, so record the "
                        "trace first and intervene in a separate forward."
                    )
                patched = intervention(self.name, self.layer_index, latent, timestep)
                if patched is not None:
                    if patched.shape != latent.shape:
                        raise ValueError(
                            f"Latent intervention on {self.name} returned shape {tuple(patched.shape)}, "
                            f"expected {tuple(latent.shape)}"
                        )
                    latent = patched
                    y_hat = self.transcoder.decoder(latent.to(dtype=self.transcoder.decoder.weight.dtype))
            else:
                self.context.record_trace(self.name, self.layer_index, preactivation, latent, timestep)
            self.context.record_latent(self.name, self.layer_index, latent, timestep)
            return y_hat.to(dtype=x.dtype)

        y = self.original_mlp(x)
        self.context.record(self.name, self.layer_index, x, y, timestep)

        if self.context.mode == "probe" and self.transcoder is not None:
            with torch.no_grad():
                _y_hat, latent, preactivation = self.transcoder(x, timestep, return_preactivation=True)
                self.context.record_trace(self.name, self.layer_index, preactivation, latent, timestep)
                self.context.record_latent(self.name, self.layer_index, latent, timestep)

        return y


def _core_pi05_model(policy_or_model: nn.Module) -> nn.Module:
    return getattr(policy_or_model, "model", policy_or_model)


def _module_parent(root: nn.Module, module_name: str) -> tuple[nn.Module, str]:
    parts = module_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def _target_mlps(model: nn.Module) -> list[tuple[str, int, nn.Module]]:
    targets: list[tuple[str, int, nn.Module]] = []
    for name, module in model.named_modules():
        if isinstance(module, WrappedActionExpertMLP):
            continue
        match = ACTION_EXPERT_MLP_RE.fullmatch(name)
        if match is None:
            continue
        targets.append((name, int(match.group(1)), module))
    return targets


def infer_mlp_d_model(module: nn.Module) -> int:
    """Infer the hidden width of a Gemma-style MLP module."""
    for attr in ("down_proj", "up_proj", "gate_proj"):
        projection = getattr(module, attr, None)
        if isinstance(projection, nn.Linear):
            if attr == "down_proj":
                return projection.out_features
            return projection.in_features
    raise ValueError(f"Could not infer d_model for MLP module {module.__class__.__name__}")


def list_pi05_action_expert_mlp_targets(policy_or_model: nn.Module) -> list[ActionExpertMLPTarget]:
    """List action-expert MLPs targeted by DifFRACT-style transcoders."""
    model = _core_pi05_model(policy_or_model)
    return [
        ActionExpertMLPTarget(
            name=name,
            layer_index=layer_index,
            module=module,
            d_model=infer_mlp_d_model(module),
        )
        for name, layer_index, module in _target_mlps(model)
    ]


def _patch_timestep_methods(model: nn.Module, context: Pi05TranscoderContext) -> None:
    if not hasattr(model, "_pi05_mi_original_forward"):
        model._pi05_mi_original_forward = model.forward  # type: ignore[attr-defined]
    if not hasattr(model, "_pi05_mi_original_denoise_step"):
        model._pi05_mi_original_denoise_step = model.denoise_step  # type: ignore[attr-defined]
    if not hasattr(model, "_pi05_mi_original_sample_noise"):
        model._pi05_mi_original_sample_noise = model.sample_noise  # type: ignore[attr-defined]

    original_forward = model._pi05_mi_original_forward  # type: ignore[attr-defined]
    original_denoise_step = model._pi05_mi_original_denoise_step  # type: ignore[attr-defined]
    original_sample_noise = model._pi05_mi_original_sample_noise  # type: ignore[attr-defined]

    def forward_with_timestep(_self: nn.Module, *args, **kwargs):
        timestep = kwargs.get("time")
        if timestep is None:
            if len(args) < 7:
                raise TypeError("PI05Pytorch.forward wrapper could not find positional argument `time`")
            timestep = args[6]
        with context.use_timestep(timestep):
            return original_forward(*args, **kwargs)

    def denoise_step_with_timestep(_self: nn.Module, *args, **kwargs):
        timestep = kwargs.get("timestep")
        if timestep is None:
            if len(args) < 4:
                raise TypeError("PI05Pytorch.denoise_step wrapper could not find positional argument `timestep`")
            timestep = args[3]
        x_t = kwargs.get("x_t")
        if x_t is None:
            if len(args) < 3:
                raise TypeError("PI05Pytorch.denoise_step wrapper could not find positional argument `x_t`")
            x_t = args[2]
        context.record_diffusion("denoise_input", x_t, timestep)
        with context.use_timestep(timestep):
            velocity = original_denoise_step(*args, **kwargs)
        context.record_diffusion("denoise_velocity", velocity, timestep)
        return velocity

    def sample_noise_with_trace(_self: nn.Module, *args, **kwargs):
        noise = original_sample_noise(*args, **kwargs)
        context.record_diffusion("initial_noise", noise, None)
        return noise

    model.forward = types.MethodType(forward_with_timestep, model)  # type: ignore[method-assign]
    model.denoise_step = types.MethodType(denoise_step_with_timestep, model)  # type: ignore[method-assign]
    model.sample_noise = types.MethodType(sample_noise_with_trace, model)  # type: ignore[method-assign]
    model._pi05_mi_timestep_context = context  # type: ignore[attr-defined]


def install_pi05_action_expert_wrappers(
    policy_or_model: nn.Module,
    *,
    context: Pi05TranscoderContext | None = None,
    transcoders: dict[str, TimeConditionedTranscoder] | None = None,
    mode: TranscoderMode = "train",
) -> tuple[Pi05TranscoderContext, list[str]]:
    """Replace Pi0.5 action-expert MLP modules with wrapper modules.

    Args:
        policy_or_model: Either a ``PI05Policy`` or its underlying ``PI05Pytorch`` model.
        context: Optional shared context. A new one is created when omitted.
        transcoders: Optional mapping from target MLP module name to transcoder.
        mode: Initial context mode.

    Returns:
        The shared context and the list of wrapped module names.
    """
    model = _core_pi05_model(policy_or_model)
    if context is None:
        context = Pi05TranscoderContext(mode=mode)
    else:
        context.mode = mode
    transcoders = transcoders or {}

    wrapped_names: list[str] = []
    for name, layer_index, module in _target_mlps(model):
        parent, attr = _module_parent(model, name)
        setattr(
            parent,
            attr,
            WrappedActionExpertMLP(
                name=name,
                layer_index=layer_index,
                original_mlp=module,
                context=context,
                transcoder=transcoders.get(name),
            ),
        )
        wrapped_names.append(name)

    _patch_timestep_methods(model, context)
    return context, wrapped_names


In [ ]:
# @title Definitions 2 — Streaming Statistics And Checkpoint Evidence
"""Checkpoint and streaming fidelity diagnostics; no LeRobot or simulator imports."""

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path
import numpy as np
import torch



@dataclass(frozen=True)
class AuditConfig:
    tasks: tuple[int, ...] = (0, 7, 8, 9)
    starts: tuple[int, ...] = (0, 1, 2)
    seeds_per_start: int = 1
    inference_steps: int = 10
    replacement_layers: tuple[int, ...] = (0, 8, 17)
    replace_all: bool = True
    action_prefix: int = 10
    seed: int = 20261112

    def cases(self):
        if (not self.tasks or not self.starts or not self.replacement_layers or
            len(set(self.tasks)) != len(self.tasks) or len(set(self.starts)) != len(self.starts) or
            len(set(self.replacement_layers)) != len(self.replacement_layers) or
            min((*self.tasks, *self.starts, *self.replacement_layers)) < 0 or
            min(self.seeds_per_start, self.inference_steps, self.action_prefix) < 1):
            raise ValueError('Audit IDs must be unique/nonnegative; counts must be positive')
        return [dict(case_id=f'task{t}_init{i}_noise{n}', task_id=t, init_state_id=i,
                     noise_index=n, scene_seed=self.seed+1009*t+i,
                     noise_seed=self.seed+100003*t+101*i+n)
                for t in self.tasks for i in self.starts for n in range(self.seeds_per_start)]


def audit_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    encoded = json.dumps(payload, indent=2, sort_keys=True, allow_nan=False)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(encoded+'\n')
    temporary.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def tensor_sha256(value):
    a = value.detach().float().cpu().contiguous().numpy()
    return hashlib.sha256(str(a.shape).encode()+a.tobytes()).hexdigest()


def batch_sha256(batch):
    digest=hashlib.sha256()
    def visit(value):
        if isinstance(value,torch.Tensor):
            digest.update(str(value.dtype).encode()); digest.update(tensor_sha256(value).encode())
        elif isinstance(value,dict):
            for key in sorted(value):digest.update(str(key).encode());visit(value[key])
        elif isinstance(value,(list,tuple)):
            digest.update(str(len(value)).encode())
            for item in value:visit(item)
        elif isinstance(value,np.ndarray):
            digest.update(str((value.shape,value.dtype)).encode());digest.update(value.tobytes())
        else:digest.update(repr(value).encode())
    visit(batch)
    return digest.hexdigest()


def checkpoint_evidence(checkpoint):
    """Inspect optimizer evidence without treating checkpoint names as updates."""
    configs, states = checkpoint['configs'], checkpoint['state_dicts']
    if set(configs) != set(states) or not configs:
        raise ValueError('Checkpoint configurations and dictionaries do not match')
    layers = []
    for name, cfg in configs.items():
        weights = states[name]
        bad = [key for key, value in weights.items() if not bool(torch.isfinite(value).all())]
        decoder = weights['decoder.weight'].float()
        norms = decoder.norm(dim=0)
        finite_number=lambda v:float(v) if bool(torch.isfinite(v)) else None
        layers.append(dict(name=name, layer=int(name.split('.')[-2]), config=cfg,
            finite=not bad, nonfinite_tensors=bad, parameters=sum(v.numel() for v in weights.values()),
            decoder_norm_min=finite_number(norms.min()), decoder_norm_mean=finite_number(norms.mean()),
            decoder_norm_max=finite_number(norms.max())))
    optimizer = checkpoint.get('optimizer_state_dict')
    parameter_rows = []
    groups = optimizer.get('param_groups', []) if optimizer else []
    ids = [p for group in groups for p in group['params']]
    for pid in ids:
        state = optimizer.get('state', {}).get(pid, {})
        step = state.get('step')
        step = float(step) if step is not None else None
        if step is not None and not np.isfinite(step):
            step = None
        parameter_rows.append(dict(parameter_id=pid, step=step))
    steps = [r['step'] for r in parameter_rows if r['step'] is not None]
    if optimizer is None:
        update_status = 'unknown_optimizer_not_saved'
    elif not optimizer.get('state'):
        update_status = 'no_optimizer_updates_recorded'
    elif ids and len(steps) == len(ids) and min(steps) > 0:
        update_status = 'positive_steps_for_all_saved_optimizer_parameters'
    else:
        update_status = 'incomplete_or_zero_step_evidence'
    history = checkpoint.get('metrics_history', [])
    train_rows = [r for r in history if 'event' not in r and 'minibatches' in r]
    bad_metrics=[dict(row=i,key=k,value=str(v)) for i,row in enumerate(history)
                 for k,v in row.items() if isinstance(v,float) and not np.isfinite(v)]
    last_metrics=[{k:(str(v) if isinstance(v,float) and not np.isfinite(v) else v)
                  for k,v in row.items()} for row in history[-5:]]
    return dict(collection_mode=checkpoint.get('collection_mode'),
        collection_feed_forwards=checkpoint.get('feed_forward'),
        policy_path=checkpoint.get('policy_path'), num_inference_steps=checkpoint.get('num_inference_steps'),
        lambda_l1=checkpoint.get('lambda_l1'), layers=layers,
        all_dictionary_weights_finite=all(r['finite'] for r in layers),
        optimizer_update_status=update_status, optimizer_parameter_count=len(ids),
        optimizer_positive_step_parameters=sum(s is not None and s > 0 for s in [r['step'] for r in parameter_rows]),
        optimizer_step_min=min(steps) if steps else None, optimizer_step_max=max(steps) if steps else None,
        optimizer_steps=parameter_rows,
        history_rows=len(history), training_rows=len(train_rows),
        history_total_layer_minibatches=sum(r.get('minibatches') or 0 for r in train_rows
                                           if r.get('minibatches') is None or np.isfinite(r['minibatches'])),
        history_rows_without_updates=sum(not r.get('minibatches') for r in train_rows),
        last_metrics=last_metrics, nonfinite_history_metrics=bad_metrics,
        limitations=['Optimizer evidence is not convergence or semantic validity.',
                    'Optimizer IDs are not assigned to layer names without an original parameter-order manifest.',
                    'Collection step numbers are not optimizer step counts.',
                    'No training split, teacher weight hash or runtime version is inferred from the filename.'])


class FitBucket:
    """Mergeable sufficient statistics: all records counted, no activation ring."""
    def __init__(self):
        self.n = 0
        self.mean = None
        self.m2 = self.sse = self.energy = self.l0 = 0.0
        self.x_energy = self.max_abs_x = self.max_abs_y = 0.0
        self.fp16_overflow_elements = 0
        self.calls = 0
        self.t_min, self.t_max = None, None

    def update(self, x, y, predicted, z, timestep):
        arrays = [v.detach().float().reshape(-1, v.shape[-1]) for v in (x, y, predicted, z)]
        if any(not bool(torch.isfinite(v).all()) for v in arrays):
            raise ValueError('Non-finite activation, reconstruction or sparse code; refusing a fidelity score')
        xx, yy, pp, zz = arrays
        if xx.shape != yy.shape or yy.shape != pp.shape or zz.shape[0] != yy.shape[0]:
            raise ValueError('Unmatched activation records')
        yd = yy.double()
        count, mean = len(yy), yd.mean(0).cpu().numpy()
        batch_m2 = float(((yd-yd.mean(0))**2).sum())
        self.merge_moments(count, mean, batch_m2)
        self.sse += float(((pp.double()-yd)**2).sum())
        self.energy += float(yd.square().sum())
        self.x_energy += float(xx.double().square().sum())
        self.l0 += float((zz > 0).sum())
        self.max_abs_x = max(self.max_abs_x, float(xx.abs().max()))
        self.max_abs_y = max(self.max_abs_y, float(yy.abs().max()))
        self.fp16_overflow_elements += int((xx.abs()>65504).sum())+int((yy.abs()>65504).sum())
        self.calls += 1
        self.t_min = timestep if self.t_min is None else min(self.t_min, timestep)
        self.t_max = timestep if self.t_max is None else max(self.t_max, timestep)
        return (zz > 0).sum(0).cpu().numpy().astype(np.int64)

    def merge_moments(self, n, mean, m2):
        if not self.n:
            self.n, self.mean, self.m2 = n, mean.copy(), m2
            return
        delta = mean-self.mean
        total = self.n+n
        self.m2 += m2+float(delta @ delta)*self.n*n/total
        self.mean += delta*n/total
        self.n = total

    def merge(self, other):
        if not other.n:
            return
        self.merge_moments(other.n, other.mean, other.m2)
        for key in ('sse', 'energy', 'l0', 'x_energy', 'fp16_overflow_elements', 'calls'):
            setattr(self, key, getattr(self,key)+getattr(other,key))
        for key in ('max_abs_x', 'max_abs_y'):
            setattr(self,key,max(getattr(self,key),getattr(other,key)))
        self.t_min = other.t_min if self.t_min is None else min(self.t_min,other.t_min)
        self.t_max = other.t_max if self.t_max is None else max(self.t_max,other.t_max)

    def row(self, common_variance=None):
        variance = self.m2/self.n if self.n else 0
        return dict(records=self.n, calls=self.calls, sse=self.sse, target_energy=self.energy,
            mse_per_vector=self.sse/self.n if self.n else None,
            own_variance_nmse=self.sse/self.m2 if self.m2>1e-12 else None,
            common_variance_nmse=self.sse/self.n/common_variance if self.n and common_variance and common_variance>1e-12 else None,
            variance_sum=variance, relative_rmse=(self.sse/self.energy)**.5 if self.energy>1e-12 else None,
            l0_mean=self.l0/self.n if self.n else None,
            input_rms=(self.x_energy/self.n/len(self.mean))**.5 if self.n else None,
            target_rms=(self.energy/self.n/len(self.mean))**.5 if self.n else None,
            max_abs_x=self.max_abs_x, max_abs_y=self.max_abs_y,
            fp16_overflow_elements=self.fp16_overflow_elements, t_min=self.t_min,t_max=self.t_max)


class AuditAccumulator:
    def __init__(self):
        self.buckets, self.occupancy, self.first_output_hashes = {}, {}, {}

    def update(self, mode, layer, x, y, predicted, z, t, call):
        # Fixed width flow-time bins, independent of sampled/trajectory call order.
        tau_bin = min(9,max(0,int(np.ceil(float(t)*10-1e-6))-1))
        key = f'{mode}|{layer}|{tau_bin}'
        bucket = self.buckets.setdefault(key,FitBucket())
        counts = bucket.update(x,y,predicted,z,float(t))
        occupancy_key = f'{mode}|{layer}'
        self.occupancy[occupancy_key] = self.occupancy.get(occupancy_key,np.zeros_like(counts))+counts
        if call == 0:
            self.first_output_hashes[occupancy_key] = tensor_sha256(y)

    def save(self, path):
        arrays = {f'occupancy::{k}':v for k,v in self.occupancy.items()}
        scalars = {}
        for key,b in self.buckets.items():
            arrays[f'mean::{key}'] = b.mean
            scalars[key] = {k:v for k,v in vars(b).items() if k!='mean'}
        arrays['metadata'] = np.array(json.dumps(dict(buckets=scalars,hashes=self.first_output_hashes)))
        temporary = Path(path).with_suffix('.tmp')
        with temporary.open('wb') as handle:
            np.savez_compressed(handle, **arrays)
        temporary.replace(path)

    @classmethod
    def load(cls, path):
        result = cls()
        with np.load(path,allow_pickle=False) as data:
            metadata=json.loads(str(data['metadata']))
            result.first_output_hashes=metadata['hashes']
            for key,values in metadata['buckets'].items():
                b=FitBucket(); b.__dict__.update(values); b.mean=data[f'mean::{key}'].copy()
                result.buckets[key]=b
            result.occupancy={k.removeprefix('occupancy::'):data[k].copy() for k in data.files if k.startswith('occupancy::')}
        return result

    def merge(self, other):
        for key,b in other.buckets.items():
            self.buckets.setdefault(key,FitBucket()).merge(b)
        for key,counts in other.occupancy.items():
            self.occupancy[key]=self.occupancy.get(key,np.zeros_like(counts))+counts


def action_error(reference, other, prefix):
    a,b=np.asarray(reference,dtype=float),np.asarray(other,dtype=float)
    if a.shape!=b.shape or a.ndim!=2 or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError('Invalid or mismatched action chunks')
    def stats(a,b):
        difference=b-a; norm=np.linalg.norm(a)
        return dict(rmse=float(np.sqrt(np.mean(difference**2))),
            relative_l2=float(np.linalg.norm(difference)/norm) if norm>1e-12 else None,
            max_abs=float(np.max(np.abs(difference))))
    return dict(full=stats(a,b),prefix=stats(a[:prefix],b[:prefix]),
                prefix_steps=min(prefix,len(a)),exactly_equal=bool(np.array_equal(a,b)))


def task_interval(rows, field, *, seed=0):
    by_task={}
    for row in rows:
        value=row.get(field)
        if value is not None and np.isfinite(value):
            by_task.setdefault(row['task_id'],[]).append(value)
    means=[float(np.mean(v)) for v in by_task.values()]
    ci=None
    if len(means)>=3:
        boot=np.random.default_rng(seed).choice(means,size=(4000,len(means)),replace=True).mean(1)
        ci=np.quantile(boot,[.025,.975]).tolist()
    return dict(task_balanced_mean=float(np.mean(means)) if means else None,
                task_bootstrap_95ci=ci,tasks=len(means),observations=sum(map(len,by_task.values())))


def summarize_audit(root, config, checkpoint_report):
    root=Path(root); merged=AuditAccumulator(); cases=[]; action_rows=[]; case_fit=[]
    planned=config.cases()
    for case in planned:
        folder=root/'cases'/case['case_id']
        if not (folder/'result.json').exists():
            continue
        row=json.loads((folder/'result.json').read_text())
        if row['case']!=case:
            raise ValueError('Saved case does not match protocol')
        current=AuditAccumulator.load(folder/'fit_stats.npz')
        merged.merge(current); cases.append(row)
        per_case={}
        for key,b in current.buckets.items():
            mode,layer,_=key.split('|'); per_case.setdefault((mode,layer),FitBucket()).merge(b)
        for (mode,layer),b in per_case.items():
            if mode!='trajectory':
                native=per_case[('trajectory',layer)]
                case_fit.append(dict(task_id=case['task_id'],case_id=case['case_id'],layer=int(layer),mode=mode,
                                     mse_difference=native.sse/native.n-b.sse/b.n))
        for mode,metrics in row['actions'].items():
            action_rows.append(dict(**case,mode=mode,rmse=metrics['prefix']['rmse'],
                                    relative_l2=metrics['prefix']['relative_l2']))
    pooled={}
    for key,b in merged.buckets.items():
        mode,layer,_=key.split('|'); pooled.setdefault((mode,layer),FitBucket()).merge(b)
    fit=[]
    for (mode,layer),b in sorted(pooled.items()):
        native=pooled[('trajectory',layer)]
        counts=merged.occupancy[f'{mode}|{layer}']
        fit.append(dict(mode=mode,layer=int(layer),**b.row(native.m2/native.n),
            dictionary_width=len(counts),never_fired_in_sample=int((counts==0).sum()),
            never_fired_fraction=float((counts==0).mean()),
            occupancy_scope='Observed sample only; never-fired does not establish a dead feature'))
    per_time=[]
    for key,b in sorted(merged.buckets.items()):
        mode,layer,tau_bin=key.split('|'); native=pooled[('trajectory',layer)]
        per_time.append(dict(mode=mode,layer=int(layer),tau_bin=int(tau_bin),
                             tau_low=int(tau_bin)/10,tau_high=(int(tau_bin)+1)/10,**b.row(native.m2/native.n)))
    actions={mode:dict(relative_l2=task_interval([r for r in action_rows if r['mode']==mode],'relative_l2'),
                       rmse=task_interval([r for r in action_rows if r['mode']==mode],'rmse'))
             for mode in sorted({r['mode'] for r in action_rows})}
    contrasts=[]
    for layer in sorted({r['layer'] for r in case_fit}):
        native=pooled[('trajectory',str(layer))]; denominator=native.m2/native.n
        for mode in ('noise_grid','noise_sampled'):
            rows=[dict(r,delta=r['mse_difference']/denominator if denominator>1e-12 else None)
                  for r in case_fit if r['layer']==layer and r['mode']==mode]
            contrasts.append(dict(layer=layer,comparison='trajectory_minus_'+mode,
                                  **task_interval(rows,'delta')))
    report=dict(planned_cases=len(planned),completed_cases=len(cases),
        completed_starts=len({(r['case']['task_id'],r['case']['init_state_id']) for r in cases}),
        completed_tasks=len({r['case']['task_id'] for r in cases}),
        status='complete' if len(cases)==len(planned) else 'partial',fit=fit,per_time=per_time,actions=actions,
        reconstruction_contrasts=contrasts,
        first_step_control_failures=sum(not c['equal_within_repeat_floor'] for row in cases for c in row['first_step_controls']),
        checkpoint_update_status=checkpoint_report['optimizer_update_status'],
        limitations=['Diagnostic cohort uses initial simulator observations, not full rollouts or occlusions.',
            'States are not established as held out from policy/transcoder pretraining.',
            'Current runtime fidelity does not establish fidelity under the original training runtime.',
            'All modes use the same trajectory-derived layer variance denominator for common_variance_nmse.',
            'Relative action errors are in normalized policy action coordinates; actions are not executed.',
            'Few task clusters make bootstrap intervals imprecise; no universal fidelity threshold is assumed.',
            'No feature attribution, steerability, collision prevention or task success is certified.'])
    audit_json(root/'report.json',report)
    lines=['# Transcoder diagnostic evidence','',f"Completed **{len(cases)}/{len(planned)} cases**, {report['completed_starts']} starts, {report['completed_tasks']} tasks.",'',
           '| Layer | Inference nMSE | Fresh-noise/grid nMSE | Random-time nMSE | Inference active features/token |',
           '|---|---:|---:|---:|---:|']
    show=lambda value:'n/a' if value is None else f'{value:.5g}'
    lookup={(r['mode'],r['layer']):r for r in fit}
    for layer in sorted({r['layer'] for r in fit}):
        values=[lookup[(mode,layer)]['common_variance_nmse'] for mode in ('trajectory','noise_grid','noise_sampled')]
        lines.append('| '+str(layer)+' | '+' | '.join(map(show,values))+' | '+show(lookup[('trajectory',layer)]['l0_mean'])+' |')
    lines += ['', 'nMSE uses a common denominator from original inference activations. Lower reconstruction error is better.',
        '', '| Replacement | Prefix relative action error | 95% task-bootstrap interval | Cases |', '|---|---:|---|---:|']
    for mode,metrics in actions.items():
        value=metrics['relative_l2']; interval=value['task_bootstrap_95ci']
        lines.append(f"| {mode} | {show(value['task_balanced_mean'])} | {interval if interval else 'insufficient tasks'} | {value['observations']} |")
    lines += ['','Optimizer evidence: '+checkpoint_report['optimizer_update_status'],
              f"First-step noise-grid controls outside repeat floor: {report['first_step_control_failures']}",
              'Paired reconstruction differences and task-bootstrap intervals: report.json → reconstruction_contrasts.',
              '',*['- '+v for v in report['limitations']]]
    if report['first_step_control_failures']:
        lines += ['', '**First-step pairing control failed: treat the distribution comparison as inconclusive until resolved.**']
    (root/'summary.md').write_text('\n'.join(lines)+'\n')
    return report


In [ ]:
# @title Definitions 3 — Simulator Snapshots And Direct Audit Runner
import argparse
import logging
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable
import numpy as np
import torch

@dataclass
class Harness:
    vec_env: Any
    inner_env: Any
    prompt: str
    preprocess_observation: Callable[[dict], dict]
    # Absent on a --list-objects pass, which needs the scene but not the policy.
    policy: Any = None
    preprocessor: Any = None
    env_preprocessor: Any = None


def ensure_libero_config() -> Path | None:
    """Write LIBERO's config.yaml so importing it cannot block on input().

    ``libero/libero/__init__.py`` prompts interactively ("Do you want to specify
    a custom path for the dataset folder?") the first time it is imported if
    ``~/.libero/config.yaml`` is missing. In any non-interactive process that
    raises ``EOFError`` before a single useful line runs.

    Writing the same defaults LIBERO would have written makes the import silent.
    The keys mirror ``get_default_path_dict`` in that module.
    """
    import importlib.util

    config_dir = Path(os.environ.get("LIBERO_CONFIG_PATH") or (Path.home() / ".libero"))
    config_file = config_dir / "config.yaml"
    if config_file.exists():
        return config_file

    # find_spec on the top-level package does not execute the submodule that
    # holds the prompt, so this is safe to call before the real import.
    spec = importlib.util.find_spec("libero")
    if spec is None or not spec.origin:
        return None
    benchmark_root = Path(spec.origin).parent / "libero"

    try:
        import yaml
    except ImportError:
        return None

    payload = {
        "benchmark_root": str(benchmark_root),
        "bddl_files": str(benchmark_root / "bddl_files"),
        "init_states": str(benchmark_root / "init_files"),
        "datasets": str(benchmark_root.parent / "datasets"),
        "assets": str(benchmark_root / "assets"),
    }
    config_dir.mkdir(parents=True, exist_ok=True)
    config_file.write_text(yaml.safe_dump(payload), encoding="utf-8")
    print(f"wrote LIBERO config {config_file} (benchmark_root={benchmark_root})", flush=True)
    return config_file


def build_harness(args: argparse.Namespace, *, load_policy: bool = True) -> Harness:
    """Build the LIBERO env, and the policy stack only when it will be used.

    Listing the scene's objects needs the simulator but not Pi0.5, and loading
    the policy means a multi-GB download, so the discovery pass skips it.
    """
    ensure_libero_config()

    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.envs.configs import LiberoEnv
    from lerobot.envs.factory import make_env, make_env_pre_post_processors
    from lerobot.policies.factory import make_policy, make_pre_post_processors
    from lerobot.scripts.lerobot_eval import preprocess_observation

    env_cfg = LiberoEnv(task=args.suite, task_ids=[args.task_id], fps=20)
    print(f"building LIBERO env {args.suite} task {args.task_id}", flush=True)
    envs = make_env(env_cfg, n_envs=1, use_async_envs=False)
    vec_env = envs[args.suite][args.task_id]
    inner_env = vec_env.envs[0] if hasattr(vec_env, "envs") else vec_env

    harness = Harness(
        vec_env=vec_env,
        inner_env=inner_env,
        prompt=args.prompt or "",
        preprocess_observation=preprocess_observation,
    )
    if not load_policy:
        print("skipping policy load (discovery pass)", flush=True)
        return harness

    try:
        print(f"loading policy {args.policy_path}", flush=True)
        policy_cfg = PreTrainedConfig.from_pretrained(
            args.policy_path,
            cache_dir=os.environ.get("HF_HUB_CACHE"),
            local_files_only=os.environ.get("HF_HUB_OFFLINE") == "1",
        )
        policy_cfg.pretrained_path = Path(args.policy_path)
        policy_cfg.device = args.device
        policy_cfg.dtype = args.policy_dtype
        policy_cfg.compile_model = False
        policy_cfg.gradient_checkpointing = False
        policy_cfg.n_action_steps = 10

        # LeRobot swallows a failed state-dict load and hands back random weights;
        # record what load_state_dict reported and refuse to measure on anything
        # but a complete load.
        install_load_recorder()
        policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg, rename_map={})
        assert_weights_loaded(policy, source=str(args.policy_path))
        policy.eval()
        for parameter in policy.parameters():
            parameter.requires_grad_(False)

        preprocessor, _ = make_pre_post_processors(
            policy_cfg=policy_cfg,
            pretrained_path=args.policy_path,
            preprocessor_overrides={
                "device_processor": {"device": str(policy.config.device)},
                "rename_observations_processor": {"rename_map": {}},
            },
        )
        harness.policy = policy
        harness.preprocessor = preprocessor
        harness.env_preprocessor, _ = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)
        return harness
    except BaseException:
        try:
            vec_env.close()
        except Exception:
            logging.exception('LIBERO cleanup failed after policy loading failed')
        raise


def _add_batch_axis(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: _add_batch_axis(item) for key, item in value.items()}
    if isinstance(value, np.ndarray):
        return value[None, ...]
    if isinstance(value, (int, float)):
        return np.asarray([value])
    return value


def _infer_prompt(harness: Harness, observation: dict[str, Any]) -> str:
    for candidate in (observation.get("task"), getattr(harness.inner_env, "task_description", None)):
        if isinstance(candidate, (list, tuple)) and candidate:
            return str(candidate[0])
        if isinstance(candidate, str) and candidate:
            return candidate
    language = getattr(getattr(harness.inner_env, "_env", None), "language_instruction", None)
    return str(language) if language else ""


"""Main-thread checkpoint audit. Embedded literally in the diagnostic notebook."""
import inspect
import hashlib
import json
import random
import time
from dataclasses import asdict
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import torch



def stage_audit_checkpoint(source, cache_dir):
    """Copy once to local disk for mmap; validate the cached file on every run."""
    source, cache_dir=Path(source),Path(cache_dir)
    if not source.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {source}')
    cache_dir.mkdir(parents=True,exist_ok=True)
    signature=dict(path=str(source.resolve()),size=source.stat().st_size,mtime_ns=source.stat().st_mtime_ns)
    receipt=cache_dir/'receipt.json'; local=cache_dir/'checkpoint.pt'
    old=json.loads(receipt.read_text()) if receipt.exists() else None
    if old and old['source']==signature and local.is_file() and local.stat().st_size==signature['size']:
        digest=file_sha256(local)
        if digest==old['sha256']:
            print('Reusing verified local checkpoint:',local,flush=True)
            return local,digest
    import hashlib
    digest=hashlib.sha256(); total=0; temporary=local.with_suffix('.partial')
    with source.open('rb') as reader,temporary.open('wb') as writer:
        for block in iter(lambda:reader.read(16*1024*1024),b''):
            writer.write(block); digest.update(block); total+=len(block)
            if total%(256*1024*1024)==0:
                print(f'Checkpoint copied: {total/2**30:.2f}/{signature["size"]/2**30:.2f} GiB',flush=True)
    if total!=signature['size'] or source.stat().st_mtime_ns!=signature['mtime_ns']:
        raise RuntimeError('Checkpoint changed during copy; rerun after the writer finishes')
    temporary.replace(local)
    digest=digest.hexdigest(); audit_json(receipt,dict(source=signature,sha256=digest))
    return local,digest


class AuditHooks:
    """Observe native MLPs and optionally replace them; retain exact time/context."""
    def __init__(self, policy, transcoders):
        self.policy,self.transcoders=policy,transcoders
        self.targets=list_pi05_action_expert_mlp_targets(policy)
        if set(transcoders)!={t.name for t in self.targets}:
            raise ValueError('Policy MLP names and checkpoint layers must match exactly')
        self.original=policy.model.denoise_step
        self.signature=inspect.signature(self.original)
        self.handles=[]; self.mode=None; self.acc=None; self.index=0; self.replacements=set()
        self.active=False; self.replace_calls=0; self.first_y={}; self.times=[]
        for target in self.targets:
            self.handles.append(target.module.register_forward_hook(self._hook(target)))
        def denoise(*args,**kwargs):
            bound=self.signature.bind(*args,**kwargs)
            if self.mode=='noise_grid':
                bound.arguments['x_t']=self.noise.clone()
            elif self.mode=='noise_sampled':
                bound.arguments['x_t']=self.draw_noise[self.index].clone()
                bound.arguments['timestep']=self.draw_times[self.index:self.index+1]
            self.t=float(bound.arguments['timestep'].detach().float().reshape(-1)[0])
            self.times.append(self.t); self.active=True
            try:
                return self.original(*bound.args,**bound.kwargs)
            finally:
                self.active=False; self.index+=1
        policy.model.denoise_step=denoise

    def _hook(self,target):
        def hook(module,inputs,output):
            if not self.active:
                raise RuntimeError('Action MLP called outside denoise_step; unsupported policy path')
            if self.index==0:
                self.first_y[target.layer_index]=output.detach().float().cpu().clone()
            measure=self.mode in ('trajectory','noise_grid','noise_sampled')
            replace=target.layer_index in self.replacements
            if not measure and not replace:
                return output
            tc=self.transcoders[target.name]
            t=torch.full((inputs[0].shape[0],),self.t,device=inputs[0].device,dtype=torch.float32)
            predicted,z=tc(inputs[0],t)
            if predicted.shape!=output.shape:
                raise ValueError('Transcoder output shape does not match native MLP')
            if measure:
                self.acc.update(self.mode,target.layer_index,inputs[0],output,predicted,z,self.t,self.index)
            if replace:
                self.replace_calls+=1
                if not bool(torch.isfinite(predicted).all()):
                    raise ValueError('Non-finite replacement output')
                return predicted.to(dtype=output.dtype)
            return output
        return hook

    def predict(self,batch,noise,steps,*,mode,acc,replacements=(),draw_times=None,draw_noise=None):
        before=batch_sha256(batch)
        self.mode,self.acc,self.noise=mode,acc,noise
        self.replacements=set(replacements); self.index=0; self.replace_calls=0
        self.first_y={}; self.times=[]; self.draw_times,self.draw_noise=draw_times,draw_noise
        self.policy.reset()
        with torch.inference_mode():
            action=self.policy.predict_action_chunk(batch,noise=noise.clone(),num_steps=steps)
        if self.index!=steps or self.replace_calls!=steps*len(self.replacements):
            raise RuntimeError('Denoising/replacement call count mismatch; refusing incomplete evidence')
        if batch_sha256(batch)!=before:
            raise RuntimeError('Policy mutated the shared observation batch; refusing unmatched comparisons')
        return action[0].detach().float().cpu().numpy()

    def close(self):
        self.policy.model.denoise_step=self.original
        for handle in self.handles:
            handle.remove()


def run_transcoder_audit(config,checkpoint_path,root,*,source_digest,runtime_report,
                         policy_path='lerobot/pi05_libero_finetuned',minutes=30,phase='all',
                         cache_dir='/content/pi05-audit-checkpoint',device='cuda'):
    """Save each complete case, resume exact protocols, and preserve native errors."""
    if phase not in ('metadata','all') or minutes<=0:
        raise ValueError('Use phase metadata/all and a positive collection budget')
    planned=config.cases(); root=Path(root); root.mkdir(parents=True,exist_ok=True)
    local,digest=stage_audit_checkpoint(checkpoint_path,cache_dir)
    checkpoint=torch.load(local,map_location='cpu',mmap=True,weights_only=True)
    old_manifest=root/'manifest.json'
    if old_manifest.exists():
        old=json.loads(old_manifest.read_text())
        if old['checkpoint_sha256']!=digest or old['source_digest']!=source_digest:
            raise ValueError('Checkpoint or notebook definitions changed; choose a new RUN_NAME')
    evidence=checkpoint_evidence(checkpoint)
    evidence.update(checkpoint_sha256=digest,source_path=str(checkpoint_path),local_path=str(local),
                    source_digest=source_digest)
    audit_json(root/'checkpoint_audit.json',evidence)
    print(json.dumps({k:evidence[k] for k in ('collection_mode','collection_feed_forwards',
        'optimizer_update_status','optimizer_step_min','optimizer_step_max','all_dictionary_weights_finite')},indent=2),flush=True)
    if phase=='metadata':
        return evidence
    if not evidence['all_dictionary_weights_finite']:
        raise ValueError('Checkpoint contains non-finite dictionary weights; see checkpoint_audit.json')
    layer_ids={r['layer'] for r in evidence['layers']}
    if not set(config.replacement_layers)<=layer_ids:
        raise ValueError('Requested replacement layer is absent from checkpoint')
    manifest=dict(config=asdict(config),source_digest=source_digest,checkpoint_sha256=digest,
                  runtime=runtime_report,policy_path=str(policy_path),
                  scope='Initial simulator observations; action predictions only; no executed interventions')
    manifest=json.loads(json.dumps(manifest))
    manifest_path=root/'manifest.json'
    if manifest_path.exists() and json.loads(manifest_path.read_text())!=manifest:
        raise ValueError('Diagnostic protocol/runtime/checkpoint changed. Choose a new RUN_NAME; saved cases are preserved.')
    audit_json(manifest_path,manifest)
    if all((root/'cases'/c['case_id']/'result.json').exists() for c in planned):
        return summarize_audit(root,config,evidence)
    base=None; active=None; hooks=None
    args=SimpleNamespace(suite='libero_spatial',task_id=config.tasks[0],prompt='',
                         policy_path=str(policy_path),device=device,policy_dtype='bfloat16')
    try:
        base=build_harness(args); policy=base.policy
        teacher_report=assert_weights_loaded(policy,source=str(policy_path))
        audit_json(root/'teacher_load.json',teacher_report)
        base.vec_env.close()
        transcoders={}
        for name,cfg in checkpoint['configs'].items():
            tc=TimeConditionedTranscoder(TimeConditionedTranscoderConfig(**cfg))
            tc.load_state_dict(checkpoint['state_dicts'][name],strict=True)
            tc.to(device=device,dtype=torch.float32).eval()
            for parameter in tc.parameters(): parameter.requires_grad_(False)
            transcoders[name]=tc
        del checkpoint
        hooks=AuditHooks(policy,transcoders)
        deadline=time.monotonic()+minutes*60
        for case in planned:
            folder=root/'cases'/case['case_id']
            if (folder/'result.json').exists():
                print('Reusing',case['case_id'],flush=True); continue
            if time.monotonic()>=deadline:
                print('Collection budget reached between cases; rerun to continue.',flush=True); break
            print('Auditing',case['case_id'],flush=True)
            random.seed(case['scene_seed']); np.random.seed(case['scene_seed'])
            args.task_id=case['task_id']; active=build_harness(args,load_policy=False)
            inner=active.inner_env
            if inner._init_states is None or case['init_state_id']>=len(inner._init_states):
                raise ValueError('Requested initial state unavailable; no modulo wrapping')
            inner.init_state_id=case['init_state_id']
            observation,_=inner.reset(seed=case['scene_seed'])
            prompt=_infer_prompt(active,observation)
            if not prompt: raise ValueError('No task instruction found')
            obs=base.preprocess_observation(_add_batch_axis(observation)); obs['task']=[prompt]
            batch=base.preprocessor(base.env_preprocessor(obs))
            generator=torch.Generator(device=device).manual_seed(case['noise_seed'])
            shape=(1,policy.model.config.chunk_size,policy.model.config.max_action_dim)
            noise=torch.randn(shape,generator=generator,device=device,dtype=torch.float32)
            draw_noise=torch.randn((config.inference_steps,*shape),generator=generator,device=device,dtype=torch.float32)
            # Model.sample_time does not accept a generator; isolate its global RNG use.
            with torch.random.fork_rng(devices=[torch.device(device).index or 0] if str(device).startswith('cuda') else []):
                torch.manual_seed(case['noise_seed']+1)
                draw_times=policy.model.sample_time(config.inference_steps,torch.device(device))
            acc=AuditAccumulator(); actions={}; arrays={}; first={}; time_draws={}
            for mode in ('trajectory','baseline_repeat','noise_grid','noise_sampled'):
                arrays[mode]=hooks.predict(batch,noise,config.inference_steps,mode=mode,acc=acc,
                                          draw_times=draw_times,draw_noise=draw_noise)
                first[mode]=dict(hooks.first_y); time_draws[mode]=list(hooks.times)
            actions['baseline_repeat']=action_error(arrays['trajectory'],arrays['baseline_repeat'],config.action_prefix)
            controls=[]
            for layer in sorted(layer_ids):
                native=first['trajectory'][layer]
                repeat=float((native-first['baseline_repeat'][layer]).abs().max())
                grid=float((native-first['noise_grid'][layer]).abs().max())
                controls.append(dict(layer=layer,repeat_max_abs=repeat,noise_grid_first_max_abs=grid,
                                     equal_within_repeat_floor=grid<=max(1e-6,repeat*5)))
            modes={f'layer_{layer}':[layer] for layer in config.replacement_layers}
            if config.replace_all: modes['all_layers']=sorted(layer_ids)
            for mode,layers in modes.items():
                arrays[mode]=hooks.predict(batch,noise,config.inference_steps,mode='replacement',acc=acc,replacements=layers)
                actions[mode]=action_error(arrays['trajectory'],arrays[mode],config.action_prefix)
            for layer in layer_ids:
                for mode in ('trajectory','noise_grid','noise_sampled'):
                    buckets=[b for k,b in acc.buckets.items() if k.startswith(f'{mode}|{layer}|')]
                    if sum(b.calls for b in buckets)!=config.inference_steps or sum(b.n for b in buckets)!=shape[1]*config.inference_steps:
                        raise RuntimeError('Missing activation records; refusing truncated statistics')
            folder.mkdir(parents=True,exist_ok=True)
            acc.save(folder/'fit_stats.npz')
            np.savez_compressed(folder/'actions.npz',**arrays)
            np.savez_compressed(folder/'initial_images.npz',**observation['pixels'])
            image_hashes={k:hashlib.sha256(np.asarray(v).tobytes()).hexdigest() for k,v in observation['pixels'].items()}
            # Readback verification precedes the result.json completion marker.
            loaded=AuditAccumulator.load(folder/'fit_stats.npz')
            if sum(b.n for b in loaded.buckets.values())!=sum(b.n for b in acc.buckets.values()):
                raise RuntimeError('Saved activation counts differ')
            audit_json(folder/'result.json',dict(case=case,prompt=prompt,initial_image_sha256=image_hashes,
                noise_sha256=tensor_sha256(noise),batch_sha256=batch_sha256(batch),time_draws=time_draws,actions=actions,
                first_step_controls=controls,grid_initial_time=time_draws['noise_grid'][0],
                native_initial_time=time_draws['trajectory'][0]))
            active.vec_env.close(); active=None
            completed=sum((root/'cases'/c['case_id']/'result.json').exists() for c in planned)
            audit_json(root/'state.json',dict(status='collecting',completed=completed,planned=len(planned)))
            print(f'Saved {completed}/{len(planned)} cases',flush=True)
        report=summarize_audit(root,config,evidence)
        audit_json(root/'state.json',dict(status=report['status'],completed=report['completed_cases'],planned=len(planned)))
        if (root/'failure.json').exists(): (root/'failure.json').unlink()
        return report
    except BaseException as exc:
        audit_json(root/'failure.json',dict(error_type=type(exc).__name__,message=str(exc)))
        raise
    finally:
        if hooks is not None: hooks.close()
        if active is not None:
            try: active.vec_env.close()
            except Exception: pass
        if base is not None and hooks is None:
            try: base.vec_env.close()
            except Exception: pass

AUDIT_NOTEBOOK_VERSION = 'b02ec246822dd828a5b0441673cbf8103fa235cb2865c78f5610093d3876274a'


In [ ]:
# @title Run Or Resume Diagnostics In This Notebook Thread
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown
if 'AUDIT_NOTEBOOK_VERSION' not in globals(): raise RuntimeError('Run the three Definitions cells first')
packages={}
for name in ('torch','transformers','lerobot','hf-libero','mujoco','numpy'):
    try: packages[name]=version(name)
    except PackageNotFoundError: packages[name]=None
parse_ids=lambda value:tuple(int(v.strip()) for v in value.split(',') if v.strip())
config=AuditConfig(tasks=parse_ids(TASK_IDS),starts=parse_ids(INIT_STATE_IDS),seeds_per_start=NOISE_DRAWS,
    inference_steps=INFERENCE_STEPS,replacement_layers=parse_ids(REPLACEMENT_LAYERS),replace_all=REPLACE_ALL)
print('Notebook definitions:',AUDIT_NOTEBOOK_VERSION[:12])
print('Planned cases:',len(config.cases()))
runtime_report=dict(packages=packages,snapshots=snapshots,
                    gpu=torch.cuda.get_device_name(0) if PHASE=='all' else None)
report=run_transcoder_audit(config,CHECKPOINT_PATH,RUN_DIR,source_digest=AUDIT_NOTEBOOK_VERSION,
    runtime_report=runtime_report,policy_path=snapshots.get(POLICY_REPO,POLICY_REPO),
    minutes=COLLECTION_MINUTES,phase=PHASE)
if PHASE=='all' and (RUN_DIR/'summary.md').exists(): display(Markdown((RUN_DIR/'summary.md').read_text()))
else:
    lines=['| Layer | Width | Features | Finite weights | Mean decoder norm |', '|---|---:|---:|---|---:|']
    for row in report['layers']:
        cfg=row['config']; features=cfg.get('d_features') or cfg['d_model']*cfg.get('expansion_factor',16)
        lines.append(f"| {row['layer']} | {cfg['d_model']} | {features} | {row['finite']} | {row['decoder_norm_mean']} |")
    display(Markdown('\n'.join(lines)))
    print('Full metadata and optimizer counters:',RUN_DIR/'checkpoint_audit.json')


In [ ]:
# @title Read Saved Results And Export A Small Evidence Packet
import zipfile
from IPython.display import display, Markdown, FileLink
summary=RUN_DIR/'summary.md'
if summary.exists(): display(Markdown(summary.read_text()))
for name in ('checkpoint_audit.json','report.json','state.json','failure.json'):
    if (RUN_DIR/name).exists(): print(name,'→',RUN_DIR/name)
packet=Path('/content')/(RUN_NAME+'-evidence.zip')
with zipfile.ZipFile(packet,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file() and (path.suffix in ('.json','.md') or path.name=='actions.npz'):
            archive.write(path,path.relative_to(RUN_DIR))
print('Full streaming statistics remain on Drive in cases/*/fit_stats.npz')
print('Small evidence packet:',packet)
display(FileLink(str(packet)))
# To download explicitly: from google.colab import files; files.download(str(packet))
